# 🚀 Kaggle Free GPU LLM Runner (Ollama + Cloudflare Tunnel)

Notebook ini menjalankan model LLM besar (misal **Qwen 2.5 27B GGUF**) secara gratis di **GPU Tesla T4 (2x 15GB)** Kaggle, dengan akses publik permanen via **Cloudflare Tunnel**.

In [ ]:
# ================= CONFIGURATION ================
# Ganti dengan model HuggingFace GGUF yang diinginkan
MODEL_TAG = "hf.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF:Q4_K_M"

# Masukkan Tunnel Token Cloudflare Zero Trust lo di sini
# Kosongkan ("") jika ingin menggunakan temporary URL trycloudflare.com gratisan
CLOUDFLARE_TUNNEL_TOKEN = "" 

# Ukuran context window (default Qwen 27B 32k, di-boost ke 64k agar aman untuk Agent / OpenCode)
CONTEXT_WINDOW = 65536
# ================================================

### 1. Install Zstandard & Ollama

In [ ]:
!sudo apt-get update -y && sudo apt-get install -y zstd curl
!curl -fsSL https://ollama.com/install.sh | sh

### 2. Jalankan Ollama Server (Background + CORS Unlocked)

In [ ]:
import subprocess, time, os

env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"
env["OLLAMA_HOST"] = "0.0.0.0:11434"

ollama_proc = subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(5)
print("✅ Ollama daemon is running with CORS unlocked!")

### 3. Pull Model & Patch Context Window (64k / 65536)

In [ ]:
import requests, json

print(f"⬇️ Pulling model: {MODEL_TAG}...")
subprocess.run(["ollama", "pull", MODEL_TAG], check=True)

print(f"⚙️ Patching context window to {CONTEXT_WINDOW} tokens...")
patch_res = requests.post(
    "http://localhost:11434/api/create",
    json={
        "model": MODEL_TAG,
        "from": MODEL_TAG,
        "parameters": {"num_ctx": CONTEXT_WINDOW}
    }
)
print("Patch Status:", patch_res.json())

### 4. Install Cloudflared & Expose Public URL (Keep-Alive)

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!sudo dpkg -i cloudflared-linux-amd64.deb

import subprocess, re, sys

if CLOUDFLARE_TUNNEL_TOKEN.strip():
    print("🔗 Starting Permanent Cloudflare Named Tunnel...")
    subprocess.run(["cloudflared", "tunnel", "--no-autoupdate", "run", "--token", CLOUDFLARE_TUNNEL_TOKEN.strip()])
else:
    print("🌐 Starting Ephemeral Quick Tunnel (trycloudflare.com)...")
    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:11434", "--http-host-header", "localhost"],
        stderr=subprocess.PIPE,
        text=True
    )
    for line in iter(tunnel_proc.stderr.readline, ""):
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            print(f"\n🔥 PUBLIC URL OLLAMA:\n{match.group(0)}\n", flush=True)
            break
    tunnel_proc.wait()